# NF-APACT convergence video (in vivo mouse liver)

Re-runs NF-APACT on the in vivo mouse data with the settings used in `figure_realdata.ipynb` (16 delays, SIREN with 0 hidden layers and 256 features, no TV, 10 epochs, batch size 64, lr $5\times10^{-3}$) and records the SOS after every epoch. At each recorded epoch, the full initial pressure is obtained by deconvolving all patches with the current SOS, as in the final deconvolution step of `reconstruction.py`. The video starts at the random initialization (epoch 0), giving `n_epochs + 1` frames. Left: reconstructed initial pressure. Right: reconstructed SOS.

In [1]:
import os
import sys

sys.path.insert(0, '../../')

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video
from matplotlib import animation
from matplotlib.colors import Normalize
from torch.optim import Adam
from tqdm import tqdm

from kwave.ktransducer import kWaveGrid
from models.coord_sos_pact import CoordSOSPACT
from models.das import DelayAndSum
from utils.dataio import load_config, prepare_data
from utils.dataset import get_data_loader
from utils.simulations import get_water_sos
from utils.visualization import standardize

FONT_WEIGHT = 'semibold'

plt.rcParams.update({'font.weight': FONT_WEIGHT, 'font.family': 'DejaVu Serif'})

In [2]:
n_delays = 16
hidden_layers, hidden_features = 0, 256
lam_tv = 0.0
n_epochs = 10
batch_size = 64
lr = 5e-3
FPS = 3
HOLD_LAST = 1.0        # seconds the final frame is held
SHOW_EPOCH = False     # print the epoch number on the initial-pressure panel
SEED = 0
DATA_PATH = '../../data/'

### Config and data

In [3]:
config = load_config('../../config.yaml')
bps, tps = config['basic_params'], config['in_vivo']

sinogram, EIR, ring_error = prepare_data(DATA_PATH, tps['sinogram'], tps['EIR'], tps['ring_error'])
kgrid = kWaveGrid([tps['Nx'], tps['Ny']], [bps['dx'], bps['dy']])
x_vec, y_vec = kgrid.x_vec+tps['x_c']*bps['dx'], kgrid.y_vec+tps['y_c']*bps['dy']
v0 = get_water_sos(tps['T'])
l_patch, N_patch = bps['l_patch'], bps['N_patch']
delays = torch.linspace(-8e-4, 8e-4, n_delays).cuda().view(-1,1,1)
sinogram = torch.from_numpy(sinogram[:,tps['t0']:]).cuda()
ring_error = torch.from_numpy(ring_error).cuda()

### DAS stack

In [4]:
das = DelayAndSum(R_ring=bps['R_ring'], N_transducer=bps['N_transducer'], T_sample=bps['T_sample'], x_vec=x_vec, y_vec=y_vec, mode='zero')
das.cuda()
das.eval()

das_stack = []
with torch.no_grad():
    for d_delay in tqdm(delays, desc='Delay-and-sum'):
        das_stack.append(das(sinogram=sinogram, v0=v0, d_delay=d_delay, ring_error=ring_error))
das_stack = torch.stack(das_stack, dim=0)
das_stack = (das_stack - das_stack.mean()) / das_stack.std()

torch.manual_seed(SEED)
data_loader = get_data_loader(das_stack=das_stack, batch_size=batch_size, l_patch=l_patch, N_patch=N_patch, stride=bps['stride'], R_body=tps['R_body'], shuffle=True)
# A separate unshuffled loader for the snapshots, so recording frames does not change the training shuffle.
snapshot_loader = get_data_loader(das_stack=das_stack, batch_size=batch_size, l_patch=l_patch, N_patch=N_patch, stride=bps['stride'], R_body=tps['R_body'], shuffle=False)
print(f'{len(data_loader.dataset)} patches, {len(data_loader)} iterations per epoch')

Delay-and-sum:   0%|          | 0/16 [00:00<?, ?it/s]

Delay-and-sum:   6%|▋         | 1/16 [00:00<00:01,  8.50it/s]

Delay-and-sum:  19%|█▉        | 3/16 [00:00<00:01, 10.43it/s]

Delay-and-sum:  31%|███▏      | 5/16 [00:00<00:01, 10.75it/s]

Delay-and-sum:  44%|████▍     | 7/16 [00:00<00:00, 10.89it/s]

...

Delay-and-sum:  69%|██████▉   | 11/16 [00:01<00:00, 10.99it/s]

Delay-and-sum:  81%|████████▏ | 13/16 [00:01<00:00, 10.70it/s]

Delay-and-sum:  94%|█████████▍| 15/16 [00:01<00:00, 10.54it/s]

Delay-and-sum: 100%|██████████| 16/16 [00:01<00:00, 10.61it/s]

621 patches, 10 iterations per epoch


### Run NF-APACT with the trajectory recorded

In [5]:
nf_apact = CoordSOSPACT(rep='SIREN', n_delays=n_delays, hidden_layers=hidden_layers, hidden_features=hidden_features, pos_encoding=False, N_freq=0, lam_tv=lam_tv,
                        x_vec=kgrid.x_vec, y_vec=kgrid.y_vec, R_body=tps['R_body'], v0=v0, mean=tps['mean'], std=tps['std'], N_patch=N_patch, l_patch=l_patch, angle_range=(0, 2*torch.pi))
nf_apact.cuda()
optimizer = Adam(params=nf_apact.parameters(), lr=lr)


def snapshot():
    """Deconvolve all patches with the current SOS and return the full initial pressure and SOS."""
    nf_apact.eval()
    with torch.no_grad():
        nf_apact.save_sos()
        ip_rec = torch.zeros((tps['Nx'], tps['Ny'])).cuda()
        for i_list, j_list, x, y, patch_stack in snapshot_loader:
            x, y, patch_stack = x.cuda(), y.cuda(), patch_stack.cuda()
            rec_patchs, sos_rec, _, _ = nf_apact(x, y, patch_stack, delays, task='Deconv')
            for i, j, rec_patch in zip(i_list, j_list, rec_patchs):
                ip_rec[20*i:20*i+80, 20*j:20*j+80] += rec_patch.squeeze(0).squeeze(0)
    nf_apact.train()
    return ip_rec.cpu().numpy(), sos_rec.cpu().numpy()


# IP_traj[k] / SOS_traj[k] are recorded after epoch k; epoch 0 is the random initialization.
IP_traj, SOS_traj, loss_list = [], [], []
ip, sos = snapshot()
IP_traj.append(ip); SOS_traj.append(sos)

nf_apact.train()
for epoch in tqdm(range(1, n_epochs+1), desc='NF-APACT'):
    loss_batch = []
    for i_list, j_list, x, y, patch_stack in data_loader:
        x, y, patch_stack = x.cuda(), y.cuda(), patch_stack.cuda()
        _, _, data_fitting, regularization = nf_apact(x, y, patch_stack, delays)
        loss = data_fitting + regularization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_batch.append(loss.item())
    loss_list.append(np.mean(loss_batch))
    ip, sos = snapshot()
    IP_traj.append(ip); SOS_traj.append(sos)

IP_traj, SOS_traj = np.stack(IP_traj), np.stack(SOS_traj)
n_frames = len(IP_traj)
print(f'trajectory {IP_traj.shape}, {n_frames} frames (epochs 0 ... {n_epochs})')

NF-APACT:   0%|          | 0/10 [00:00<?, ?it/s]

NF-APACT:  10%|█         | 1/10 [00:05<00:46,  5.13s/it]

NF-APACT:  20%|██        | 2/10 [00:10<00:40,  5.08s/it]

NF-APACT:  30%|███       | 3/10 [00:15<00:35,  5.07s/it]

NF-APACT:  40%|████      | 4/10 [00:20<00:30,  5.06s/it]

...

NF-APACT:  80%|████████  | 8/10 [00:40<00:10,  5.06s/it]

NF-APACT:  90%|█████████ | 9/10 [00:45<00:05,  5.06s/it]

NF-APACT: 100%|██████████| 10/10 [00:50<00:00,  5.06s/it]

NF-APACT: 100%|██████████| 10/10 [00:50<00:00,  5.07s/it]

trajectory (11, 560, 560), 11 frames (epochs 0 ... 10)


### Video

In [6]:
# Colour scales as in figure_invivo: the standardized initial pressure in grey on [-3.5, 8] and the
# SOS in magma on [1500, 1580] m/s. They are fixed across frames. No titles or colour bars.
norm_IP = Normalize(vmin=-3.5, vmax=8)
norm_SOS = Normalize(vmin=tps['SOS_min'], vmax=tps['SOS_max'])

fig, (ax_ip, ax_sos) = plt.subplots(1, 2, figsize=(10, 5))
fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.04)

h_ip = ax_ip.imshow(standardize(IP_traj[0]), cmap='gray', norm=norm_IP)
h_sos = ax_sos.imshow(SOS_traj[0], cmap='magma', norm=norm_SOS)
h_txt = ax_ip.text(12, 30, '', color='white', fontsize=16, fontweight=FONT_WEIGHT) if SHOW_EPOCH else None
for ax in (ax_ip, ax_sos):
    ax.set_axis_off()


def update(i):
    h_ip.set_data(standardize(IP_traj[i]))
    h_sos.set_data(SOS_traj[i])
    if SHOW_EPOCH:
        h_txt.set_text(f'Epoch {i}')
    return h_ip, h_sos


# Hold the final reconstruction for HOLD_LAST seconds by repeating its frame.
frames = list(range(n_frames)) + [n_frames - 1] * int(round(HOLD_LAST * FPS))
anim = animation.FuncAnimation(fig, update, frames=frames, blit=False)
video_path = 'video_nf_apact_invivo.mp4'
anim.save(video_path, writer=animation.FFMpegWriter(fps=FPS, bitrate=2400), dpi=150,
          savefig_kwargs={'facecolor': 'white'})
# The last frame (the final reconstruction) as a still image.
update(n_frames - 1)
image_path = 'video_nf_apact_invivo_last.png'
fig.savefig(image_path, dpi=150, facecolor='white')
plt.close(fig)
print('saved', video_path, 'and', image_path)
Video(video_path, embed=True)

saved video_nf_apact_invivo.mp4 and video_nf_apact_invivo_last.png
